# NVIDIA Nemotron-PII Dataset — Loading & Processing

This notebook loads the full `nvidia/Nemotron-PII` dataset from HuggingFace, explores its structure, and prepares it for downstream PII filtering and processing work.

## 1. Install and Import Required Libraries

In [1]:
%uv pip install -q datasets pandas numpy pyarrow

/Users/lucas/Documents/sonex_hackfest/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [2]:
import ast
import gc
import json
import re

import numpy as np
import pandas as pd
from datasets import load_dataset

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)

print("Libraries loaded successfully.")


/Users/lucas/Documents/sonex_hackfest/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully.


## 2. Load the NVIDIA Nemotron-PII Dataset

Loads the full dataset from HuggingFace. The `spans` field is a Python-repr encoded string, so we parse it with `ast.literal_eval`.

In [3]:
def _parse_spans(raw) -> list[dict]:
    """Parse the spans field which may be a Python-repr string or JSON string."""
    if not isinstance(raw, str):
        return raw or []
    try:
        return ast.literal_eval(raw)
    except (ValueError, SyntaxError):
        pass
    try:
        return json.loads(raw)
    except (json.JSONDecodeError, ValueError):
        return []


# Load only the required split — avoids caching the full DatasetDict in RAM
print("Downloading nvidia/Nemotron-PII from HuggingFace (this may take a moment)...")
try:
    data = load_dataset("nvidia/Nemotron-PII", split="test")
    split = "test"
except Exception:
    data = load_dataset("nvidia/Nemotron-PII", split="train")
    split = "train"

print(f"\nUsing split: '{split}'  |  Total rows: {len(data):,}")
print(f"Column names: {data.column_names}")



Using split: 'test'  |  Total rows: 100,000
Column names: ['uid', 'domain', 'document_type', 'document_description', 'document_format', 'locale', 'text', 'spans', 'text_tagged']


In [4]:
# Convert to pandas DataFrame and parse spans into Python objects
df = data.to_pandas()
del data  # free the Arrow dataset — no longer needed
gc.collect()

# Normalise column names in case they differ across dataset versions
col_map = {}
for col in df.columns:
    if col in ("document", "text"):
        col_map[col] = "text"
    elif col in ("source", "domain"):
        col_map[col] = "domain"
df = df.rename(columns=col_map)

# Parse spans column
df["spans"] = df["spans"].apply(_parse_spans)

print(f"DataFrame shape: {df.shape}")
df.head(3)


DataFrame shape: (100000, 9)


,uid,domain,document_type,document_description,document_format,locale,text,spans,text_tagged
0,ebd01589fd75420da79eea362ce54e2c,Life,"visa application (e.g., F-1, H-1B)","A life and visa application document, such as an F-1 or H-1B form, is typically an unstructured PDF or printable wor...",unstructured,us,"Brian, born on 1963-08-08, resides at 146 County Rd 86 in Knox County, USA. As part of the visa application process...","[{'start': 0, 'end': 5, 'text': 'Brian', 'label': 'first_name'}, {'start': 15, 'end': 25, 'text': '1963-08-08', 'lab...","[Brian]first_name, born on [1963-08-08]date_of_birth, resides at [146 County Rd 86]street_address in [Knox County]co..."
1,db716ae2c78f485dadb9b87e2cd8110c,User Account and Transaction Services,Credit Card Authorization Form,"This unstructured document, titled ""Credit Card Authorization Form"" for ""User Account and Transaction Services,"" typ...",unstructured,us,This Credit Card Authorization Form authorizes the transaction for the user name: lkernan. The account number: 87425...,"[{'start': 82, 'end': 89, 'text': 'lkernan', 'label': 'user_name'}, {'start': 111, 'end': 119, 'text': '87425693', '...",This Credit Card Authorization Form authorizes the transaction for the user name: [lkernan]user_name. The account nu...
2,bde4585a9dea42cdaf8a881ef3e69167,Property,Rental Application,A Property and Rental Application is a structured document typically formatted as a multi-page PDF or fillable onlin...,structured,us,**Property and Rental Application**\n\n**Applicant Personal Information**\n\nFirst Name: Sabrina\n\nStreet Address: ...,"[{'start': 85, 'end': 92, 'text': 'Sabrina', 'label': 'first_name'}, {'start': 110, 'end': 131, 'text': '122 S West ...",**Property and Rental Application**\n\n**Applicant Personal Information**\n\nFirst Name: [Sabrina]first_name\n\nStre...


## 3. Explore Dataset Structure

In [5]:
# Schema and basic types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   uid                   100000 non-null  str   
 1   domain                100000 non-null  str   
 2   document_type         100000 non-null  str   
 3   document_description  100000 non-null  str   
 4   document_format       100000 non-null  str   
 5   locale                100000 non-null  str   
 6   text                  100000 non-null  str   
 7   spans                 100000 non-null  object
 8   text_tagged           100000 non-null  str   
dtypes: object(1), str(8)
memory usage: 238.9+ MB


In [6]:
# Span count per document
df["n_spans"] = df["spans"].apply(len)

# Explode spans into a flat label list for frequency analysis
# list comprehension is ~50-100x faster than iterrows and builds no intermediate objects
spans_df = pd.DataFrame(
    [
        {"label": span.get("label", ""), "text_snippet": span.get("text", ""), "domain": domain}
        for domain, spans in zip(df["domain"], df["spans"])
        for span in spans
    ]
)

print(f"Total span annotations: {len(spans_df):,}")
print(f"\nUnique PII label types ({spans_df['label'].nunique()}):")
print(spans_df["label"].value_counts().to_string())

del spans_df  # only needed for the frequency report above
gc.collect()


Total span annotations: 850,340

Unique PII label types (55):
label
first_name                        84043
date                              73867
last_name                         59596
company_name                      54837
email                             53930
url                               37847
occupation                        36888
time                              24506
phone_number                      23930
country                           23475
customer_id                       20502
city                              18347
state                             18318
date_of_birth                     18079
street_address                    16879
account_number                    16693
user_name                         15800
date_time                         13221
credit_debit_card                 12867
biometric_identifier              11379
medical_record_number             11098
employment_status                 11018
health_plan_beneficiary_number    10558
education_le

374

In [7]:
# Domain / source distribution
if "domain" in df.columns:
    print("Document counts by domain:")
    print(df["domain"].value_counts().to_string())
else:
    print("No 'domain' column found in this dataset version.")

# Basic text-length stats
df["text_len"] = df["text"].str.len()
df[["text_len", "n_spans"]].describe().round(1)

Document counts by domain:
domain
Banking                                  3840
User Account and Transaction Services    3794
Information Technology                   3786
Brokerage                                3762
Access Control Systems                   3728
Credit                                   3714
Disability                               3634
Life                                     3632
Marketing                                3622
Consulting                               3620
Social Science                           3576
Services                                 3574
Human Resources                          3566
Health                                   3558
Healthcare Providers                     3500
Investment                               3496
Advertising                              3464
Casualty                                 3464
Travel                                   3458
Elections                                3392
Fitness                                  3258


,text_len,n_spans
count,100000.0,100000.0
mean,958.5,8.5
std,699.0,4.6
min,45.0,0.0
25%,483.0,6.0
50%,745.0,7.0
75%,1203.0,10.0
max,14608.0,92.0


## 4. Initial Data Filtering

Separate documents by PII category. Adjust the label sets below to match your processing goals.

In [8]:
CRITICAL_PII_LABELS = {
    "ssn", "credit_debit_card", "bank_routing_number",
    "password", "pin", "cvv", "biometric_identifier",
}

ALL_PII_LABELS = {
    "person_name", "ssn", "date_of_birth", "national_id", "passport_number",
    "drivers_license", "phone_number", "email_address", "street_address",
    "postcode", "ip_address", "credit_debit_card", "bank_routing_number",
    "account_number", "swift_bic", "cvv", "pin", "password", "api_key",
    "biometric_identifier", "employee_id", "username",
    "medical_record_number", "health_insurance_id",
}

def has_label(spans: list[dict], label_set: set[str]) -> bool:
    return any(s.get("label", "") in label_set for s in spans)

# --- Remove exact duplicates (same text) ---
df_dedup = df.drop_duplicates(subset=["text"]).copy()
print(f"After dedup: {len(df_dedup):,} rows (removed {len(df) - len(df_dedup):,})")
del df  # original df no longer needed
gc.collect()

# --- Report critical vs non-critical PII counts (no full partition copies) ---
mask_critical = df_dedup["spans"].apply(lambda s: has_label(s, CRITICAL_PII_LABELS))
n_critical = int(mask_critical.sum())
print(f"\nDocuments with critical PII  : {n_critical:,}")
print(f"Documents without critical PII: {len(df_dedup) - n_critical:,}")
del mask_critical
gc.collect()


After dedup: 100,000 rows (removed 0)

Documents with critical PII  : 38,508
Documents without critical PII: 61,492


0

## 5. Handle Missing Values

In [9]:
# Audit missing values across all columns
missing = df_dedup.isnull().sum()
print("Null counts per column:")
print(missing[missing > 0].to_string() if missing.any() else "  No null values found.")

# Drop rows where the main text field is missing or empty
before = len(df_dedup)
df_clean = df_dedup[df_dedup["text"].notna() & (df_dedup["text"].str.strip() != "")].copy()
print(f"\nDropped {before - len(df_clean):,} rows with empty/null text.")
del df_dedup  # free the deduped frame; df_clean is the working copy from here on
gc.collect()

# Fill missing domain with 'unknown'
if "domain" in df_clean.columns:
    df_clean["domain"] = df_clean["domain"].fillna("unknown")

print(f"Clean dataset size: {len(df_clean):,} rows")


Null counts per column:
  No null values found.

Dropped 0 rows with empty/null text.
Clean dataset size: 100,000 rows


## 6. Normalize and Clean Text Fields

In [10]:
def normalize_text(text: str) -> str:
    """Strip leading/trailing whitespace and collapse internal whitespace runs."""
    text = text.strip()
    text = re.sub(r"\r\n|\r", "\n", text)          # Normalize line endings
    text = re.sub(r"[ \t]{2,}", " ", text)          # Collapse multiple spaces/tabs
    return text

def normalize_label(label: str) -> str:
    """Lowercase and strip the PII label string."""
    return label.strip().lower()

# Apply text normalization and drop the original column to free memory
df_clean["text_norm"] = df_clean["text"].apply(normalize_text)
df_clean.drop(columns=["text"], inplace=True)

# Normalize all labels inside the spans lists and drop original spans
def normalize_spans(spans: list[dict]) -> list[dict]:
    return [{**s, "label": normalize_label(s.get("label", ""))} for s in spans]

df_clean["spans_norm"] = df_clean["spans"].apply(normalize_spans)
df_clean.drop(columns=["spans"], inplace=True)

# Derive a flat comma-joined label string per document (handy for quick filtering)
df_clean["label_set"] = df_clean["spans_norm"].apply(
    lambda spans: ",".join(sorted({s["label"] for s in spans if s.get("label")}))
)

# Use categorical dtype for low-cardinality string columns — big RAM saving
if "domain" in df_clean.columns:
    df_clean["domain"] = df_clean["domain"].astype("category")
df_clean["label_set"] = df_clean["label_set"].astype("category")

gc.collect()
print("Sample normalized rows:")
df_clean[["text_norm", "label_set", "domain"]].head(5)


Sample normalized rows:


,text_norm,label_set,domain
0,"Brian, born on 1963-08-08, resides at 146 County Rd 86 in Knox County, USA. As part of the visa application process,...","bank_routing_number,country,county,credit_debit_card,date_of_birth,first_name,street_address",Life
1,This Credit Card Authorization Form authorizes the transaction for the user name: lkernan. The account number: 87425...,"account_number,credit_debit_card,user_name",User Account and Transaction Services
2,**Property and Rental Application**\n\n**Applicant Personal Information**\n\nFirst Name: Sabrina\n\nStreet Address: ...,"first_name,race_ethnicity,street_address",Property
3,I am enrolling in the health insurance plan. My first name is Manal and my last name is King. I was born on 1983-09-...,"date_of_birth,first_name,health_plan_beneficiary_number,last_name,street_address",Access Control Systems
4,**Passenger Details**\n\n- **First Name**: Hosana\n- **Last Name**: Garcia\n- **Email**: garciah@outlook.com\n- **Ra...,"date,email,first_name,http_cookie,last_name,mac_address,race_ethnicity,time",Travel


## 7. Export Processed Dataset

Save the cleaned dataset to disk. Use these files as the starting point for your downstream processing.

In [11]:
from pathlib import Path

OUT_DIR = Path("eval_output/pii_processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Columns to export (spans stored as JSON string for CSV compatibility)
export_cols = ["text_norm", "spans_norm", "label_set", "domain", "document_type", "n_spans", "text_len"]
available_export_cols = [c for c in export_cols if c in df_clean.columns]

# Use assign() to serialize spans_norm in-place without copying the whole DataFrame
df_export = df_clean[available_export_cols].assign(
    spans_norm=df_clean["spans_norm"].apply(json.dumps)
)

# --- CSV ---
csv_path = OUT_DIR / "nemotron_pii_full.csv"
df_export.to_csv(csv_path, index=False)
print(f"Saved CSV  → {csv_path}  ({csv_path.stat().st_size / 1024:.1f} KB)")

# --- Parquet (preserves types, better for large files) ---
try:
    parquet_path = OUT_DIR / "nemotron_pii_full.parquet"
    df_export.to_parquet(parquet_path, index=False)
    print(f"Saved Parquet → {parquet_path}  ({parquet_path.stat().st_size / 1024:.1f} KB)")
except Exception as e:
    print(f"Parquet export skipped: {e}")

del df_export
gc.collect()
print(f"\nExported {len(df_clean):,} rows with {len(available_export_cols)} columns.")


Saved CSV  → eval_output/pii_processed/nemotron_pii_full.csv  (175768.1 KB)
Saved Parquet → eval_output/pii_processed/nemotron_pii_full.parquet  (71718.5 KB)

Exported 100,000 rows with 7 columns.


## 8. LM Studio — Classify Document Types by Agent Type

Use a locally running LLM (via LM Studio's OpenAI-compatible API) to keep only document types that are the direct **input or output of one of the three patrol-swarm agent types**:

| Agent | Produces / consumes |
|---|---|
| **Coding agent** | Source code, scripts, config files, CI/CD, logs, test files, READMEs, changelogs, bug/issue reports, technical specs |
| **Email agent** | Emails, notifications, newsletters, correspondence |
| **Document agent** | Reports, memos, contracts, policies, proposals, meeting notes, invoices, HR records, audit files, agreements, guidelines, internal forms |

**Small-model strategy (9-14B 4-bit):** types are sent in batches of 25 with a short, direct system prompt. The model returns *integer indices* only — tiny output that avoids hallucinated names and JSON length issues.

In [12]:
%pip install -q openai

/Users/lucas/Documents/sonex_hackfest/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
from openai import OpenAI

# LM Studio exposes an OpenAI-compatible server — make sure it is running
LM_STUDIO_URL = "http://localhost:1234/v1"

client = OpenAI(base_url=LM_STUDIO_URL, api_key="nvidia_nvidia-nemotron-nano-9b-v2")

# Verify connection and show loaded model(s)
models = client.models.list()
model_ids = [m.id for m in models.data]
print("Connected to LM Studio.")
print("Available model(s):", model_ids)

# Use whichever model is loaded
MODEL = model_ids[0]
print(f"Using: {MODEL}")

Connected to LM Studio.
Available model(s): ['nvidia_nvidia-nemotron-nano-9b-v2', 'qwen3-4b-instruct-2507-mlx', 'smollm2-360m-instruct', 'zai-org/glm-4.6v-flash', 'qwen3-vl-4b-instruct-mlx', 'qwen/qwen3-vl-8b', 'qwen/qwen2.5-coder-14b', 'text-embedding-nomic-embed-text-v1.5']
Using: nvidia_nvidia-nemotron-nano-9b-v2


In [14]:
import pandas as pd
from pathlib import Path

# Load the already-exported processed dataset (avoids re-running the full pipeline)
_parquet = Path("eval_output/pii_processed/nemotron_pii_full.parquet")
_csv     = Path("eval_output/pii_processed/nemotron_pii_full.csv")

if _parquet.exists():
    df_processed = pd.read_parquet(_parquet)
    print(f"Loaded parquet: {_parquet}  ({len(df_processed):,} rows)")
elif _csv.exists():
    df_processed = pd.read_csv(_csv)
    print(f"Loaded CSV: {_csv}  ({len(df_processed):,} rows)")
else:
    raise FileNotFoundError(
        "Exported dataset not found. Run Section 7 (Export Processed Dataset) first."
    )

# Guard: document_type must be present — if not, the parquet is stale (exported before
# document_type was added to export_cols). Re-run Section 7 to regenerate it.
if "document_type" not in df_processed.columns:
    raise KeyError(
        "'document_type' column is missing from the exported file.\n"
        "Re-run Section 7 (Export Processed Dataset) to regenerate the parquet with this column, "
        "then re-run this cell."
    )

print(f"Columns present: {list(df_processed.columns)}")

# Extract unique document_type values — each type appears exactly once
all_doc_types: set[str] = set(df_processed["document_type"].dropna().unique())
print(f"\nUnique document_type values ({len(all_doc_types)} total) — these are what the LLM will classify:")
for t in sorted(all_doc_types):
    print(f"  {t}")


Loaded parquet: eval_output/pii_processed/nemotron_pii_full.parquet  (100,000 rows)
Columns present: ['text_norm', 'spans_norm', 'label_set', 'domain', 'document_type', 'n_spans', 'text_len']

Unique document_type values (1466 total) — these are what the LLM will classify:
  A/B Test Results
  A/B Testing
  API Deployment
  API Documentation
  API Endpoint
  API Key
  API Logging
  API Method
  API Monitoring
  API Parameter
  API Rate Limit
  API Response
  API Security
  API Specification
  API Testing
  API Version
  API Versioning
  API biometric identifier payload
  ATM card PIN mailer
  Access Control Log
  Access Token
  Accident Report
  Account Activity Report
  Account Closing Form
  Account Opening Form
  Account Registration Form
  Account Statement
  Account Summary
  Account Transfer
  Activity Log
  Ad Campaign Brief
  Ad Campaign Plan
  Ad Campaign Review
  Ad Compliance Report
  Ad Content Guidelines
  Ad Copy
  Ad Creative Brief
  Ad Creative Review
  Ad Design
  Ad P

In [15]:
import json as _json
import re as _re
import time
from openai import OpenAI

# Reinitialize client if not already defined (handles out-of-order execution)
if "client" not in dir() or not isinstance(client, OpenAI):
    client = OpenAI(base_url=LM_STUDIO_URL, api_key="nvidia_nvidia-nemotron-nano-9b-v2")
    MODEL = [m.id for m in client.models.list().data][0]
    print(f"Re-initialized client. Using: {MODEL}")

# ── Why batched + index output for 9-14B 4-bit models ────────────────────────
# 1. Small models lose coherence on prompts with hundreds of items — batches of
#    BATCH_SIZE keep the context short and focused.
# 2. Asking for *integer indices* (tiny output) instead of full strings prevents
#    the model from hallucinating new names or mis-spelling existing ones.
# 3. Short, imperative system prompt — small quantized models follow brief,
#    direct instructions more reliably than long multi-paragraph ones.
# 4. max_tokens=256 comfortably fits the index list output.
# 5. 3-attempt retry + regex integer fallback handles occasional malformed JSON.
# ── Thinking / reasoning mode ─────────────────────────────────────────────────
# Thinking is DISABLED via /no_think in the system prompt.
# Reason: with thinking ON, the model burns its token budget on <think>...</think>
# before emitting the JSON array, causing truncation and wrong results.
# Thinking adds zero benefit for a simple binary classification task like this.
# ─────────────────────────────────────────────────────────────────────────────

# BATCH_SIZE tradeoff:
#   1  → pure yes/no per item, maximum accuracy, but 1 API call per doc type (~200+ calls)
#   25 → picks from a list of 25; quality is identical for a task this simple,
#        and it runs ~25x faster. Coherence only degrades above ~50-75 items.
BATCH_SIZE = 25

SYSTEM_PROMPT = (
    "/no_think "  # disables <think> reasoning tokens for Nemotron/Qwen3 models
    "You are a strict document classifier for an AI agent-swarm evaluation dataset. "
    "The swarm contains exactly three agent types: "
    "(1) a CODING AGENT that produces or consumes source code, scripts, config files, "
    "logs, CI/CD files, test files, READMEs, changelogs, bug reports, and technical specs; "
    "(2) an EMAIL AGENT that composes or reads emails, notifications, and correspondence; "
    "(3) a DOCUMENT AGENT that drafts or edits internal business documents such as reports, "
    "memos, contracts, policies, proposals, meeting notes, invoices, HR records, audit files, "
    "agreements, guidelines, and internal forms. "
    "You will receive a numbered list of document type names. "
    "Reply with ONLY a JSON array of the integers whose document type could be directly "
    "created, read, or modified by one of those three agents. "
    "Exclude anything that is a personal consumer form, a physical ID/card, a retail financial "
    "product brochure, a recreational/lifestyle record, or medical patient paperwork. "
    "Output format: [2, 5, 11] — integers only, no explanation, no markdown fences."
)

sorted_types = sorted(all_doc_types)
batches = [sorted_types[i: i + BATCH_SIZE] for i in range(0, len(sorted_types), BATCH_SIZE)]

print(f"{len(sorted_types)} doc types → {len(batches)} batches of ≤{BATCH_SIZE}")

relevant_indices: set[int] = set()  # accumulates 0-based indices into sorted_types

for batch_num, batch in enumerate(batches):
    offset = batch_num * BATCH_SIZE
    numbered = "\n".join(f"{i + 1}. {t}" for i, t in enumerate(batch))
    user_msg = f"Classify these document types (reply with relevant item numbers only):\n{numbered}"

    for attempt in range(3):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            temperature=0.0,
            max_tokens=256,  # index list is short; 256 gives plenty of headroom
        )
        reply = resp.choices[0].message.content.strip()

        # Strip any residual <think>...</think> block just in case
        reply = _re.sub(r"<think>.*?</think>", "", reply, flags=_re.DOTALL).strip()

        # Strip optional markdown fences
        clean = _re.sub(r"```[\w]*", "", reply).strip()
        try:
            nums = _json.loads(clean)
            if not isinstance(nums, list):
                raise ValueError("not a list")
            for n in nums:
                idx = int(n) - 1  # 1-based → 0-based within batch
                if 0 <= idx < len(batch):
                    relevant_indices.add(offset + idx)
            break  # parsed successfully
        except Exception:
            # Fallback: grab all integers that look like valid item numbers
            found = [int(x) - 1 for x in _re.findall(r'\b(\d+)\b', reply)
                     if 0 < int(x) <= len(batch)]
            if found:
                for idx in found:
                    relevant_indices.add(offset + idx)
                break

            if attempt == 2:
                print(f"  ⚠ batch {batch_num + 1}: could not parse reply: {reply!r}")

    print(f"  batch {batch_num + 1}/{len(batches)} done — {len(relevant_indices)} selected so far")
    time.sleep(0.1)  # small pause so LM Studio isn't overwhelmed

print(f"\nFinished. Total relevant types selected: {len(relevant_indices)}")

1466 doc types → 59 batches of ≤25
  batch 1/59 done — 12 selected so far
  batch 2/59 done — 25 selected so far
  batch 3/59 done — 28 selected so far
  batch 4/59 done — 40 selected so far
  batch 5/59 done — 54 selected so far
  batch 6/59 done — 54 selected so far
  batch 7/59 done — 64 selected so far
  batch 8/59 done — 74 selected so far
  batch 9/59 done — 98 selected so far
  batch 10/59 done — 108 selected so far
  batch 11/59 done — 117 selected so far
  batch 12/59 done — 138 selected so far
  batch 13/59 done — 158 selected so far
  batch 14/59 done — 175 selected so far
  batch 15/59 done — 177 selected so far
  batch 16/59 done — 197 selected so far
  batch 17/59 done — 212 selected so far
  batch 18/59 done — 234 selected so far
  batch 19/59 done — 241 selected so far
  batch 20/59 done — 252 selected so far
  batch 21/59 done — 259 selected so far
  batch 22/59 done — 272 selected so far
  batch 23/59 done — 278 selected so far
  batch 24/59 done — 283 selected so far

In [16]:
# Map selected 0-based indices back to type names
matched_types = sorted(sorted_types[i] for i in relevant_indices)

print(f"Selected {len(matched_types)} relevant document types:\n")
for t in matched_types:
    print(f"  {t}")

Selected 684 relevant document types:

  A/B Testing
  API Documentation
  API Logging
  API Monitoring
  API Rate Limit
  API Response
  API Specification
  API Testing
  API Version
  API Versioning
  Access Control Log
  Access Token
  Account Statement
  Activity Log
  Ad Campaign Brief
  Ad Campaign Plan
  Ad Campaign Review
  Ad Compliance Report
  Ad Content Guidelines
  Ad Copy
  Ad Creative Brief
  Ad Creative Review
  Ad Design
  Ad Performance Report
  Ad Placement Plan
  Affidavit of No Fraud
  Affidavit of No Prior Mortgage
  Affidavit of No Unpaid Maintenance Fees
  Appraisal Report
  Asset Allocation
  Asset Statement
  Attitude Survey
  Audio Ad Script
  Audit Log
  Audit Report
  Author Contract
  Author Newsletter
  Author Presentation
  Author Report
  Author Whitepaper
  Authorization Letter
  BIC Code Document
  Background Check Authorization Form
  Bank Account Closure
  Bank Account Opening
  Bank Account Opening Form
  Bank Account Setup Form
  Bank Statement
  

In [17]:
# Apply the filter — keep only rows whose document_type was selected by the LLM
df_agent = df_processed[df_processed["document_type"].isin(matched_types)].copy()

print(f"df_processed total rows : {len(df_processed):>10,}")
print(f"df_agent  (filtered)    : {len(df_agent):>10,}  ({len(df_agent)/len(df_processed)*100:.1f}%)")
print()
print("Row counts by document_type:")
print(df_agent["document_type"].value_counts().to_string())

df_processed total rows :    100,000
df_agent  (filtered)    :     49,140  (49.1%)

Row counts by document_type:
document_type
Investment Strategy                        434
User Guide                                 310
Risk Management Plan                       286
Incident Report                            262
Claim Form                                 242
Service Level Agreement                    242
Credit Report                              240
Customer Feedback Form                     234
Performance Report                         232
User Agreement                             230
Customer Service Policy                    218
Risk Assessment                            218
User Manual                                216
Health Insurance Enrollment Form           208
Customer Agreement                         206
Account Statement                          188
Test Login Script                          168
Feasibility Study                          164
Investment Agreement       

In [18]:
from pathlib import Path
import json
import gc

AGENT_OUT_DIR = Path("eval_output/pii_agent_swarm")
AGENT_OUT_DIR.mkdir(parents=True, exist_ok=True)

# spans_norm is already a JSON string (serialized during the Section 7 export) — no re-serialization needed
df_agent_export = df_agent.copy()

csv_out     = AGENT_OUT_DIR / "agent_swarm_docs.csv"
parquet_out = AGENT_OUT_DIR / "agent_swarm_docs.parquet"

df_agent_export.to_csv(csv_out, index=False)
print(f"CSV     → {csv_out}  ({csv_out.stat().st_size/1024:.1f} KB)")

try:
    df_agent_export.to_parquet(parquet_out, index=False)
    print(f"Parquet → {parquet_out}  ({parquet_out.stat().st_size/1024:.1f} KB)")
except Exception as e:
    print(f"Parquet skipped: {e}")

del df_agent_export
gc.collect()

# Also save the list of selected doc types for reproducibility
types_out = AGENT_OUT_DIR / "selected_doc_types.json"
types_out.write_text(json.dumps(sorted(matched_types), indent=2))
print(f"Types   → {types_out}")
print(f"\nExported {len(df_agent):,} rows.")


CSV     → eval_output/pii_agent_swarm/agent_swarm_docs.csv  (91014.6 KB)
Parquet → eval_output/pii_agent_swarm/agent_swarm_docs.parquet  (37629.6 KB)
Types   → eval_output/pii_agent_swarm/selected_doc_types.json

Exported 49,140 rows.
